In [ ]:
# FireWatch AI: Temporal Analysis

In this notebook, I will analyze fire and smoke detections across consecutive video frames using my existing trained YOLO26n model.

My goal is to understand how detections change over time and estimate whether fire or smoke activity is increasing, stable, or decreasing.

I will use the extracted FIRESENSE videos and will not retrain the model.

## My plan
1. Load the existing model and verify the video dataset paths.
2. Process video frames and record detection timestamps, classes, confidence scores, and bounding boxes.
3. Compare detections across frames.
4. Estimate fire and smoke trends over time.
5. Save and verify the temporal analysis result.

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
%pip install -q ultralytics

In [4]:
from pathlib import Path
from ultralytics import YOLO

PROJECT = Path("/content/drive/MyDrive/FireWatch-AI")
MODEL_PATH = PROJECT / "runs/fire_smoke/yolo26n_fire_smoke_v1/weights/best.pt"
VIDEO_ROOT = PROJECT / "datasets/videos/firesense"
OUTPUT_DIR = PROJECT / "outputs/temporal_analysis"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert MODEL_PATH.is_file(), "Trained model checkpoint missing!"
assert (VIDEO_ROOT / "fire").is_dir(), "Fire videos folder missing!"
assert (VIDEO_ROOT / "smoke").is_dir(), "Smoke videos folder missing!"

model = YOLO(str(MODEL_PATH))

print("Model loaded:", MODEL_PATH.name)
print("Classes:", model.names)
print("Fire videos:", len(list((VIDEO_ROOT / "fire").rglob("*.avi"))))
print("Smoke videos:", len(list((VIDEO_ROOT / "smoke").rglob("*.avi"))))
print("Output folder:", OUTPUT_DIR)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Model loaded: best.pt
Classes: {0: 'fire', 1: 'smoke'}
Fire videos: 27
Smoke videos: 22
Output folder: /content/drive/MyDrive/FireWatch-AI/outputs/temporal_analysis


In [5]:
import csv

video_exts = {".avi", ".mp4", ".mov", ".mkv"}
videos = {
    "fire": next(p for p in sorted((VIDEO_ROOT / "fire").rglob("*"))
                 if p.is_file() and p.suffix.lower() in video_exts and "pos" in [x.lower() for x in p.parts]),
    "smoke": next(p for p in sorted((VIDEO_ROOT / "smoke").rglob("*"))
                  if p.is_file() and p.suffix.lower() in video_exts and "pos" in [x.lower() for x in p.parts])
}

csv_path = OUTPUT_DIR / "temporal_detections.csv"
records = []

for category, path in videos.items():
    for frame_no, result in enumerate(model.predict(
        source=str(path), conf=0.25, vid_stride=10,
        stream=True, verbose=False
    )):
        timestamp = result.path
        fps = result.speed  # inference timing only; not video FPS
        for box in result.boxes:
            records.append({
                "video_category": category,
                "video_name": path.name,
                "frame_number": frame_no,
                "class": model.names[int(box.cls[0])],
                "confidence": round(float(box.conf[0]), 4),
                "x1": round(float(box.xyxy[0][0]), 2),
                "y1": round(float(box.xyxy[0][1]), 2),
                "x2": round(float(box.xyxy[0][2]), 2),
                "y2": round(float(box.xyxy[0][3]), 2)
            })

with open(csv_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=records[0].keys())
    writer.writeheader()
    writer.writerows(records)

print("Records saved:", len(records))
print("CSV:", csv_path)
print("Videos processed:", {k: v.name for k, v in videos.items()})

Records saved: 141
CSV: /content/drive/MyDrive/FireWatch-AI/outputs/temporal_analysis/temporal_detections.csv
Videos processed: {'fire': 'posVideo1.868.avi', 'smoke': 'testpos01.817.avi'}


In [6]:
import pandas as pd
import cv2

df = pd.read_csv(csv_path)
df["timestamp_sec"] = 0.0

for category, path in videos.items():
    cap = cv2.VideoCapture(str(path))
    fps = cap.get(cv2.CAP_PROP_FPS)
    cap.release()
    if fps <= 0:
        raise ValueError(f"Could not read FPS: {path.name}")

    mask = df["video_name"] == path.name
    df.loc[mask, "timestamp_sec"] = df.loc[mask, "frame_number"] * 10 / fps

df.to_csv(csv_path, index=False)

df["box_area"] = (df["x2"] - df["x1"]) * (df["y2"] - df["y1"])
summary = df.groupby(["video_name", "class"]).agg(
    detections=("confidence", "size"),
    avg_confidence=("confidence", "mean"),
    avg_box_area=("box_area", "mean"),
    first_time=("timestamp_sec", "min"),
    last_time=("timestamp_sec", "max")
).reset_index()

print(summary.round(2).to_string(index=False))
print("\nUpdated CSV:", csv_path)

       video_name class  detections  avg_confidence  avg_box_area  first_time  last_time
posVideo1.868.avi  fire         114            0.49       3763.37        0.00      28.00
testpos01.817.avi  fire           1            0.34      27307.53       13.75      13.75
testpos01.817.avi smoke          26            0.74      47831.47        1.67      15.00

Updated CSV: /content/drive/MyDrive/FireWatch-AI/outputs/temporal_analysis/temporal_detections.csv


In [8]:
import pandas as pd

df = pd.read_csv(csv_path)

# Recalculate bounding-box area from saved coordinates.
df["box_area"] = (df["x2"] - df["x1"]) * (df["y2"] - df["y1"])
df["time_window"] = (df["timestamp_sec"] // 5).astype(int) * 5

trend = df.groupby(["video_name", "class", "time_window"]).agg(
    detections=("confidence", "size"),
    avg_confidence=("confidence", "mean"),
    avg_box_area=("box_area", "mean")
).reset_index()

def estimate_trend(group):
    group = group.sort_values("time_window")
    if len(group) < 2:
        return "Insufficient data"

    first, last = group["avg_box_area"].iloc[[0, -1]]
    change = (last - first) / max(first, 1)

    if change > 0.20:
        return "Increasing"
    if change < -0.20:
        return "Decreasing"
    return "Stable"

summary = trend.groupby(["video_name", "class"]).apply(
    estimate_trend, include_groups=False
).reset_index(name="estimated_trend")

print(summary.to_string(index=False))

       video_name class   estimated_trend
posVideo1.868.avi  fire        Increasing
testpos01.817.avi  fire Insufficient data
testpos01.817.avi smoke            Stable


In [9]:
import cv2
import pandas as pd

frame_records = []

for category, path in videos.items():
    cap = cv2.VideoCapture(str(path))
    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_no = 0
    sampled_no = 0

    while True:
        ok, frame = cap.read()
        if not ok:
            break

        if frame_no % 10 == 0:
            result = model.predict(frame, conf=0.25, verbose=False)[0]

            for cls in ["fire", "smoke"]:
                boxes = [
                    b for b in result.boxes
                    if model.names[int(b.cls[0])] == cls
                ]
                frame_records.append({
                    "video_name": path.name,
                    "category": category,
                    "frame_number": frame_no,
                    "timestamp_sec": round(frame_no / fps, 2) if fps else 0,
                    "class": cls,
                    "detections": len(boxes),
                    "avg_confidence": (
                        sum(float(b.conf[0]) for b in boxes) / len(boxes)
                        if boxes else 0
                    ),
                    "avg_box_area": (
                        sum(
                            float((b.xyxy[0][2] - b.xyxy[0][0]) *
                                  (b.xyxy[0][3] - b.xyxy[0][1]))
                            for b in boxes
                        ) / len(boxes) if boxes else 0
                    )
                })
            sampled_no += 1

        frame_no += 1

    cap.release()

frame_df = pd.DataFrame(frame_records)
frame_csv = OUTPUT_DIR / "temporal_frame_summary.csv"
frame_df.to_csv(frame_csv, index=False)

print("Frame records:", len(frame_df))
print("Saved:", frame_csv)

Frame records: 142
Saved: /content/drive/MyDrive/FireWatch-AI/outputs/temporal_analysis/temporal_frame_summary.csv


In [10]:
import pandas as pd

frame_df = pd.read_csv(OUTPUT_DIR / "temporal_frame_summary.csv")

def classify_trend(group):
    group = group.sort_values("timestamp_sec")
    if len(group) < 3:
        return "Insufficient data"

    # Compare average box area in the first and second halves.
    mid = len(group) // 2
    first = group.iloc[:mid]["avg_box_area"].mean()
    second = group.iloc[mid:]["avg_box_area"].mean()

    # Treat missing detections as zero area.
    change = (second - first) / max(first, 1)

    if change > 0.20:
        return "Increasing"
    if change < -0.20:
        return "Decreasing"
    return "Stable"

summary = frame_df.groupby(["video_name", "class"]).apply(
    classify_trend, include_groups=False
).reset_index(name="estimated_trend")

print(summary.to_string(index=False))

summary.to_csv(OUTPUT_DIR / "temporal_trend_summary.csv", index=False)
print("\nSaved:", OUTPUT_DIR / "temporal_trend_summary.csv")

       video_name class estimated_trend
posVideo1.868.avi  fire      Increasing
posVideo1.868.avi smoke          Stable
testpos01.817.avi  fire          Stable
testpos01.817.avi smoke      Increasing

Saved: /content/drive/MyDrive/FireWatch-AI/outputs/temporal_analysis/temporal_trend_summary.csv


In [11]:
import pandas as pd

frame_df = pd.read_csv(OUTPUT_DIR / "temporal_frame_summary.csv")

print("Total frame records:", len(frame_df))
print("Missing values:\n", frame_df.isna().sum())
print("\nDetection-free samples:", (frame_df["detections"] == 0).sum())
print("\nVideo/class coverage:")
print(frame_df.groupby(["video_name", "class"]).size())

Total frame records: 142
Missing values:
 video_name        0
category          0
frame_number      0
timestamp_sec     0
class             0
detections        0
avg_confidence    0
avg_box_area      0
dtype: int64

Detection-free samples: 86

Video/class coverage:
video_name         class
posVideo1.868.avi  fire     30
                   smoke    30
testpos01.817.avi  fire     41
                   smoke    41
dtype: int64


In [12]:
import pandas as pd

frame_df = pd.read_csv(OUTPUT_DIR / "temporal_frame_summary.csv")
frame_df = frame_df.sort_values(["video_name", "class", "frame_number"])

frame_df["detected"] = frame_df["detections"] > 0

# Start a new run whenever detection status changes.
frame_df["run_id"] = (
    frame_df.groupby(["video_name", "class"])["detected"]
    .transform(lambda s: s.ne(s.shift()).cumsum())
)

persistence = frame_df[frame_df["detected"]].groupby(
    ["video_name", "class", "run_id"]
).agg(
    consecutive_samples=("frame_number", "size"),
    start_sec=("timestamp_sec", "min"),
    end_sec=("timestamp_sec", "max")
).reset_index()

summary = persistence.groupby(["video_name", "class"]).agg(
    detection_runs=("run_id", "count"),
    longest_run=("consecutive_samples", "max"),
    average_run=("consecutive_samples", "mean")
).reset_index()

print(summary.to_string(index=False))
persistence.to_csv(OUTPUT_DIR / "detection_persistence.csv", index=False)
print("\nSaved:", OUTPUT_DIR / "detection_persistence.csv")

       video_name class  detection_runs  longest_run  average_run
posVideo1.868.avi  fire               1           30         30.0
testpos01.817.avi smoke               4           22          6.5

Saved: /content/drive/MyDrive/FireWatch-AI/outputs/temporal_analysis/detection_persistence.csv


In [13]:
import pandas as pd

df = pd.read_csv(OUTPUT_DIR / "temporal_frame_summary.csv")
df["time_window"] = (df["timestamp_sec"] // 5).astype(int) * 5

windows = df.groupby(["video_name", "class", "time_window"]).agg(
    detections=("detections", "sum"),
    confidence=("avg_confidence", "mean"),
    box_area=("avg_box_area", "mean")
).reset_index()

# Smooth short-term fluctuations using a 3-window rolling average.
windows = windows.sort_values(["video_name", "class", "time_window"])
windows["smooth_area"] = windows.groupby(
    ["video_name", "class"]
)["box_area"].transform(lambda s: s.rolling(3, min_periods=1).mean())

def trend_label(g):
    g = g.sort_values("time_window")
    if len(g) < 3:
        return "Insufficient data"

    first = g["smooth_area"].iloc[:len(g)//2].mean()
    second = g["smooth_area"].iloc[len(g)//2:].mean()
    change = (second - first) / max(first, 1)

    if change > 0.20:
        return "Increasing"
    if change < -0.20:
        return "Decreasing"
    return "Stable"

result = windows.groupby(["video_name", "class"]).apply(
    trend_label, include_groups=False
).reset_index(name="smoothed_trend")

windows.to_csv(OUTPUT_DIR / "temporal_smoothed_windows.csv", index=False)
result.to_csv(OUTPUT_DIR / "temporal_smoothed_trends.csv", index=False)

print(result.to_string(index=False))
print("\nSaved smoothed windows and trend summary.")

       video_name class smoothed_trend
posVideo1.868.avi  fire     Increasing
posVideo1.868.avi smoke         Stable
testpos01.817.avi  fire         Stable
testpos01.817.avi smoke     Increasing

Saved smoothed windows and trend summary.


In [14]:
import pandas as pd

frames = pd.read_csv(OUTPUT_DIR / "temporal_frame_summary.csv")
persistence = pd.read_csv(OUTPUT_DIR / "detection_persistence.csv")
trends = pd.read_csv(OUTPUT_DIR / "temporal_smoothed_trends.csv")

base = frames.groupby(["video_name", "class"]).agg(
    sampled_frames=("frame_number", "nunique"),
    detected_frames=("detections", lambda s: int((s > 0).sum())),
    avg_confidence=("avg_confidence", "mean")
).reset_index()

persist = persistence.groupby(["video_name", "class"]).agg(
    detection_runs=("run_id", "nunique"),
    longest_run=("consecutive_samples", "max")
).reset_index()

report = base.merge(persist, on=["video_name", "class"], how="left")
report = report.merge(trends, on=["video_name", "class"], how="left")

report["detection_runs"] = report["detection_runs"].fillna(0).astype(int)
report["longest_run"] = report["longest_run"].fillna(0).astype(int)
report["detection_rate"] = (
    report["detected_frames"] / report["sampled_frames"]
).round(3)
report["avg_confidence"] = report["avg_confidence"].round(3)
report["data_status"] = report["sampled_frames"].apply(
    lambda n: "Limited validation" if n < 100 else "More validation needed"
)

path = OUTPUT_DIR / "temporal_final_report.csv"
report.to_csv(path, index=False)

print(report.to_string(index=False))
print("\nSaved:", path)

       video_name class  sampled_frames  detected_frames  avg_confidence  detection_runs  longest_run smoothed_trend  detection_rate        data_status
posVideo1.868.avi  fire              30               30           0.534               1           30     Increasing           1.000 Limited validation
posVideo1.868.avi smoke              30                0           0.000               0            0         Stable           0.000 Limited validation
testpos01.817.avi  fire              41                0           0.000               0            0         Stable           0.000 Limited validation
testpos01.817.avi smoke              41               26           0.472               4           22     Increasing           0.634 Limited validation

Saved: /content/drive/MyDrive/FireWatch-AI/outputs/temporal_analysis/temporal_final_report.csv


In [15]:
import pandas as pd

report = pd.read_csv(OUTPUT_DIR / "temporal_final_report.csv")

def final_status(row):
    if row["detected_frames"] == 0:
        return "No detections"
    if row["sampled_frames"] < 10:
        return "Insufficient data"
    if row["detection_rate"] < 0.10:
        return "Intermittent detections"
    return row["smoothed_trend"]

report["final_temporal_status"] = report.apply(final_status, axis=1)

report.to_csv(OUTPUT_DIR / "temporal_final_report.csv", index=False)

print(report[[
    "video_name", "class", "detection_rate",
    "smoothed_trend", "final_temporal_status"
]].to_string(index=False))
print("\nUpdated:", OUTPUT_DIR / "temporal_final_report.csv")

       video_name class  detection_rate smoothed_trend final_temporal_status
posVideo1.868.avi  fire           1.000     Increasing            Increasing
posVideo1.868.avi smoke           0.000         Stable         No detections
testpos01.817.avi  fire           0.000         Stable         No detections
testpos01.817.avi smoke           0.634     Increasing            Increasing

Updated: /content/drive/MyDrive/FireWatch-AI/outputs/temporal_analysis/temporal_final_report.csv


In [2]:
from pathlib import Path
import cv2
import pandas as pd
from ultralytics import YOLO

PROJECT = Path("/content/drive/MyDrive/FireWatch-AI")
VIDEO_ROOT = PROJECT / "datasets/videos/firesense"
MODEL_PATH = PROJECT / "runs/fire_smoke/yolo26n_fire_smoke_v1/weights/best.pt"
OUTPUT_DIR = PROJECT / "outputs/temporal_analysis"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

model = YOLO(str(MODEL_PATH))

print("Video root exists:", VIDEO_ROOT.exists())
print("Model loaded:", MODEL_PATH.exists())

Video root exists: True
Model loaded: True


In [3]:
import pandas as pd

report = pd.read_csv(OUTPUT_DIR / "temporal_final_report.csv")

def validate_trend(row):
    if row["detected_frames"] == 0:
        return "No detections"
    if row["detection_rate"] < 0.10:
        return "Intermittent detections"
    if row["longest_run"] < 3:
        return "Insufficient persistence"
    if row["sampled_frames"] < 10:
        return "Insufficient data"
    return "Preliminary visual trend"

report["validation_status"] = report.apply(validate_trend, axis=1)
report.to_csv(OUTPUT_DIR / "temporal_final_report.csv", index=False)

print(report[[
    "video_name", "class", "detection_rate",
    "longest_run", "smoothed_trend", "validation_status"
]].to_string(index=False))
print("\nSaved:", OUTPUT_DIR / "temporal_final_report.csv")

       video_name class  detection_rate  longest_run smoothed_trend        validation_status
posVideo1.868.avi  fire           1.000           30     Increasing Preliminary visual trend
posVideo1.868.avi smoke           0.000            0         Stable            No detections
testpos01.817.avi  fire           0.000            0         Stable            No detections
testpos01.817.avi smoke           0.634           22     Increasing Preliminary visual trend

Saved: /content/drive/MyDrive/FireWatch-AI/outputs/temporal_analysis/temporal_final_report.csv


In [1]:
from pathlib import Path
import pandas as pd

OUTPUT_DIR = Path("/content/drive/MyDrive/FireWatch-AI/outputs/temporal_analysis")
save_path = OUTPUT_DIR / "positive_video_evaluation.csv"

if save_path.exists():
    df = pd.read_csv(save_path)
    print("Saved video results:", df["video_name"].nunique())
    print(df.groupby("video_category")["video_name"].nunique())
else:
    print("Evaluation CSV not found. Results may not have been saved.")

Evaluation CSV not found. Results may not have been saved.


In [3]:
from pathlib import Path
from ultralytics import YOLO

PROJECT = Path("/content/drive/MyDrive/FireWatch-AI")
VIDEO_ROOT = PROJECT / "datasets/videos/firesense"
MODEL_PATH = PROJECT / "runs/fire_smoke/yolo26n_fire_smoke_v1/weights/best.pt"
OUTPUT_DIR = PROJECT / "outputs/temporal_analysis"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert VIDEO_ROOT.exists(), "FIRESENSE video folder missing"
assert MODEL_PATH.exists(), "Trained model missing"

model = YOLO(str(MODEL_PATH))

print("Setup ready")
print("Fire videos:", len(list((VIDEO_ROOT / "fire").rglob("*.avi"))))
print("Smoke videos:", len(list((VIDEO_ROOT / "smoke").rglob("*.avi"))))

Setup ready
Fire videos: 27
Smoke videos: 22


In [ ]:
import cv2
import pandas as pd
from pathlib import Path

VIDEO_EXTS = {".avi", ".mp4", ".mov", ".mkv"}
save_path = OUTPUT_DIR / "positive_video_evaluation.csv"

completed = set()
if save_path.exists():
    old = pd.read_csv(save_path)
    completed = set(zip(old["video_category"], old["video_name"]))

for category in ["fire", "smoke"]:
    paths = sorted(
        p for p in (VIDEO_ROOT / category).rglob("*")
        if p.is_file() and p.suffix.lower() in VIDEO_EXTS
        and "pos" in [part.lower() for part in p.parts]
    )

    for path in paths:
        if (category, path.name) in completed:
            continue

        cap = cv2.VideoCapture(str(path))
        fps = cap.get(cv2.CAP_PROP_FPS)
        frame_no = sampled = 0
        detected = {"fire": 0, "smoke": 0}
        confs = {"fire": [], "smoke": []}

        while True:
            ok, frame = cap.read()
            if not ok:
                break
            if frame_no % 10 == 0:
                result = model.predict(frame, conf=0.25, verbose=False)[0]
                sampled += 1
                seen = set()
                for box in result.boxes:
                    cls = model.names[int(box.cls[0])]
                    if cls in seen or cls not in detected:
                        continue
                    seen.add(cls)
                    detected[cls] += 1
                    confs[cls].append(float(box.conf[0]))
            frame_no += 1
        cap.release()

        new_rows = [{
            "video_category": category,
            "video_name": path.name,
            "class": cls,
            "sampled_frames": sampled,
            "detected_frames": detected[cls],
            "detection_rate": round(detected[cls] / sampled, 3) if sampled else 0,
            "avg_confidence": round(sum(confs[cls]) / len(confs[cls]), 3) if confs[cls] else 0,
            "fps": round(fps, 2) if fps > 0 else None
        } for cls in detected]

        pd.DataFrame(new_rows).to_csv(
            save_path, mode="a", header=not save_path.exists(), index=False
        )
        completed.add((category, path.name))
        print(f"Saved: {category}/{path.name}")

print("\nEvaluation CSV:", save_path)
print("Videos saved:", len(completed))

Saved: fire/posVideo1.868.avi
Saved: fire/posVideo10.869.avi
Saved: fire/posVideo11.870.avi
Saved: fire/posVideo2.871.avi
Saved: fire/posVideo3.872.avi
Saved: fire/posVideo4.873.avi
Saved: fire/posVideo5.874.avi
Saved: fire/posVideo6.875.avi
Saved: fire/posVideo7.876.avi
Saved: fire/posVideo8.877.avi
Saved: fire/posVideo9.878.avi
Saved: smoke/testpos01.817.avi
Saved: smoke/testpos02.818.avi


In [1]:
import pandas as pd
from pathlib import Path

path = Path("/content/drive/MyDrive/FireWatch-AI/outputs/temporal_analysis/positive_video_evaluation.csv")

if path.exists():
    df = pd.read_csv(path)
    print("Unique videos saved:", df["video_name"].nunique())
    print(df.groupby("video_category")["video_name"].nunique())
else:
    print("CSV not found")

Unique videos saved: 13
video_category
fire     11
smoke     2
Name: video_name, dtype: int64


In [5]:
from pathlib import Path
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/FireWatch-AI")
VIDEO_ROOT = PROJECT / "datasets/videos/firesense"
save_path = PROJECT / "outputs/temporal_analysis/positive_video_evaluation.csv"

saved = pd.read_csv(save_path)
completed = set(zip(saved["video_category"], saved["video_name"]))

for category in ["fire", "smoke"]:
    paths = [
        p for p in (VIDEO_ROOT / category).rglob("*")
        if p.is_file()
        and p.suffix.lower() in {".avi", ".mp4", ".mov", ".mkv"}
        and "pos" in [part.lower() for part in p.parts]
    ]
    remaining = [p for p in paths if (category, p.name) not in completed]
    print(f"{category}: {len(remaining)} remaining")

fire: 0 remaining
smoke: 11 remaining


In [2]:
from pathlib import Path
import pandas as pd
import cv2
from ultralytics import YOLO

PROJECT = Path("/content/drive/MyDrive/FireWatch-AI")
VIDEO_ROOT = PROJECT / "datasets/videos/firesense"
MODEL_PATH = PROJECT / "runs/fire_smoke/yolo26n_fire_smoke_v1/weights/best.pt"
OUTPUT_DIR = PROJECT / "outputs/temporal_analysis"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

save_path = OUTPUT_DIR / "positive_video_evaluation.csv"
model = YOLO(str(MODEL_PATH))

print("CSV exists:", save_path.exists())
print("Saved video count:", pd.read_csv(save_path)["video_name"].nunique() if save_path.exists() else 0)
print("Setup ready.")

CSV exists: True
Saved video count: 13
Setup ready.


In [3]:
from pathlib import Path
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/FireWatch-AI")
VIDEO_ROOT = PROJECT / "datasets/videos/firesense"
save_path = PROJECT / "outputs/temporal_analysis/positive_video_evaluation.csv"

saved = pd.read_csv(save_path)
completed = set(zip(saved["video_category"], saved["video_name"]))

remaining = [
    p for p in sorted((VIDEO_ROOT / "smoke").rglob("*"))
    if p.is_file()
    and p.suffix.lower() in {".avi", ".mp4", ".mov", ".mkv"}
    and "pos" in [part.lower() for part in p.parts]
    and ("smoke", p.name) not in completed
][:2]

print("Already saved videos:", saved["video_name"].nunique())
print("Next videos:", [p.name for p in remaining])
print("Smoke videos remaining after this batch:", max(0, 11 - 2))

Already saved videos: 13
Next videos: ['testpos03.819.avi', 'testpos04.820.avi']
Smoke videos remaining after this batch: 9


In [1]:
from pathlib import Path
import pandas as pd

csv_path = Path(
    "/content/drive/MyDrive/FireWatch-AI/outputs/temporal_analysis/positive_video_evaluation.csv"
)

if csv_path.exists():
    df = pd.read_csv(csv_path)
    print("Total saved rows:", len(df))
    print("Unique videos saved:", df["video_name"].nunique())
    print("\nSaved videos:")
    print(sorted(df["video_name"].unique()))
else:
    print("CSV not found. Drive mount check karna hoga.")

Total saved rows: 26
Unique videos saved: 13

Saved videos:
['posVideo1.868.avi', 'posVideo10.869.avi', 'posVideo11.870.avi', 'posVideo2.871.avi', 'posVideo3.872.avi', 'posVideo4.873.avi', 'posVideo5.874.avi', 'posVideo6.875.avi', 'posVideo7.876.avi', 'posVideo8.877.avi', 'posVideo9.878.avi', 'testpos01.817.avi', 'testpos02.818.avi']


In [2]:
import pandas as pd
from pathlib import Path

csv_path = Path(
    "/content/drive/MyDrive/FireWatch-AI/outputs/temporal_analysis/positive_video_evaluation.csv"
)

df = pd.read_csv(csv_path)

print("Videos evaluated:", df["video_name"].nunique())
print("\nVideos by category:")
print(df.groupby("video_category")["video_name"].nunique())

print("\nEvaluation summary:")
display(
    df.groupby(["video_category", "class"])
      .agg(
          videos=("video_name", "nunique"),
          avg_detection_rate=("detection_rate", "mean"),
          avg_confidence=("avg_confidence", "mean")
      )
      .round(3)
      .reset_index()
)


Videos evaluated: 13

Videos by category:
video_category
fire     11
smoke     2
Name: video_name, dtype: int64

Evaluation summary:


,video_category,class,videos,avg_detection_rate,avg_confidence
0,fire,fire,11,0.997,0.789
1,fire,smoke,11,0.009,0.028
2,smoke,fire,2,0.482,0.298
3,smoke,smoke,2,0.317,0.374


In [3]:
from pathlib import Path
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/FireWatch-AI")
OUTPUT_DIR = PROJECT / "outputs/temporal_analysis"

csv_path = OUTPUT_DIR / "positive_video_evaluation.csv"
df = pd.read_csv(csv_path)

# Summary by video category and detected class
summary = (
    df.groupby(["video_category", "class"])
    .agg(
        videos=("video_name", "nunique"),
        avg_detection_rate=("detection_rate", "mean"),
        avg_confidence=("avg_confidence", "mean")
    )
    .round(3)
    .reset_index()
)

summary_path = OUTPUT_DIR / "positive_video_evaluation_summary.csv"
summary.to_csv(summary_path, index=False)

# Save a readable report
report_path = OUTPUT_DIR / "positive_video_evaluation_report.txt"

with open(report_path, "w") as f:
    f.write("FireWatch AI - Positive Video Evaluation Report\n")
    f.write("=" * 48 + "\n\n")
    f.write(f"Unique videos evaluated: {df['video_name'].nunique()}\n")
    f.write(f"Fire videos: {df[df['video_category'] == 'fire']['video_name'].nunique()}\n")
    f.write(f"Smoke videos: {df[df['video_category'] == 'smoke']['video_name'].nunique()}\n\n")
    f.write("Evaluation Summary:\n")
    f.write(summary.to_string(index=False))
    f.write("\n\nLimitations:\n")
    f.write("- Results cover only the currently evaluated videos.\n")
    f.write("- Detection rate is not classification accuracy.\n")
    f.write("- Smoke-category evaluation currently includes only two videos.\n")
    f.write("- Fire detections in smoke videos require further investigation.\n")

print("Summary saved:", summary_path)
print("Report saved:", report_path)
print("\n", summary.to_string(index=False))

Summary saved: /content/drive/MyDrive/FireWatch-AI/outputs/temporal_analysis/positive_video_evaluation_summary.csv
Report saved: /content/drive/MyDrive/FireWatch-AI/outputs/temporal_analysis/positive_video_evaluation_report.txt

 video_category class  videos  avg_detection_rate  avg_confidence
          fire  fire      11               0.997           0.789
          fire smoke      11               0.009           0.028
         smoke  fire       2               0.482           0.298
         smoke smoke       2               0.317           0.374


In [4]:
from pathlib import Path
import pandas as pd

OUTPUT_DIR = Path(
    "/content/drive/MyDrive/FireWatch-AI/outputs/temporal_analysis"
)

files = [
    "temporal_detections.csv",
    "temporal_frame_summary.csv",
    "detection_persistence.csv",
    "temporal_smoothed_windows.csv",
    "temporal_smoothed_trends.csv",
    "temporal_trend_summary.csv",
    "temporal_final_report.csv",
    "positive_video_evaluation.csv",
    "positive_video_evaluation_summary.csv",
    "positive_video_evaluation_report.txt",
]

for name in files:
    path = OUTPUT_DIR / name
    if path.exists():
        print(f"✅ {name} | {path.stat().st_size:,} bytes")
    else:
        print(f"❌ Missing: {name}")

✅ temporal_detections.csv | 10,053 bytes
✅ temporal_frame_summary.csv | 8,433 bytes
✅ detection_persistence.csv | 254 bytes
✅ temporal_smoothed_windows.csv | 1,310 bytes
✅ temporal_smoothed_trends.csv | 162 bytes
✅ temporal_trend_summary.csv | 163 bytes
✅ temporal_final_report.csv | 584 bytes
✅ positive_video_evaluation.csv | 1,387 bytes
✅ positive_video_evaluation_summary.csv | 164 bytes
✅ positive_video_evaluation_report.txt | 749 bytes


In [5]:
from pathlib import Path
import pandas as pd

report_path = Path(
    "/content/drive/MyDrive/FireWatch-AI/outputs/temporal_analysis/temporal_final_report.csv"
)

report = pd.read_csv(report_path)

print("Report shape:", report.shape)
display(report)

Report shape: (4, 12)


,video_name,class,sampled_frames,detected_frames,avg_confidence,detection_runs,longest_run,smoothed_trend,detection_rate,data_status,final_temporal_status,validation_status
0,posVideo1.868.avi,fire,30,30,0.534,1,30,Increasing,1.000,Limited validation,Increasing,Preliminary visual trend
1,posVideo1.868.avi,smoke,30,0,0.000,0,0,Stable,0.000,Limited validation,No detections,No detections
2,testpos01.817.avi,fire,41,0,0.000,0,0,Stable,0.000,Limited validation,No detections,No detections
3,testpos01.817.avi,smoke,41,26,0.472,4,22,Increasing,0.634,Limited validation,Increasing,Preliminary visual trend


In [6]:
from pathlib import Path
import pandas as pd

path = Path(
    "/content/drive/MyDrive/FireWatch-AI/outputs/temporal_analysis/temporal_smoothed_trends.csv"
)

trends = pd.read_csv(path)

print("Shape:", trends.shape)
display(trends)

Shape: (4, 3)


,video_name,class,smoothed_trend
0,posVideo1.868.avi,fire,Increasing
1,posVideo1.868.avi,smoke,Stable
2,testpos01.817.avi,fire,Stable
3,testpos01.817.avi,smoke,Increasing


In [7]:
from pathlib import Path
import pandas as pd

OUTPUT_DIR = Path(
    "/content/drive/MyDrive/FireWatch-AI/outputs/temporal_analysis"
)

report_path = OUTPUT_DIR / "temporal_final_report.csv"
trend_path = OUTPUT_DIR / "temporal_smoothed_trends.csv"

report = pd.read_csv(report_path)
trends = pd.read_csv(trend_path)

check = report[
    ["video_name", "class", "detected_frames", "smoothed_trend", "final_temporal_status"]
].merge(
    trends,
    on=["video_name", "class", "smoothed_trend"],
    how="left",
    indicator=True
)

print("Trend matches:", (check["_merge"] == "both").sum(), "/", len(check))
print("\nZero-detection rows:")
display(check[check["detected_frames"] == 0][
    ["video_name", "class", "detected_frames", "smoothed_trend", "final_temporal_status"]
])

Trend matches: 4 / 4

Zero-detection rows:


,video_name,class,detected_frames,smoothed_trend,final_temporal_status
1,posVideo1.868.avi,smoke,0,Stable,No detections
2,testpos01.817.avi,fire,0,Stable,No detections


In [8]:
from pathlib import Path
import pandas as pd

OUT = Path("/content/drive/MyDrive/FireWatch-AI/outputs/temporal_analysis")

required = [
    "temporal_detections.csv",
    "temporal_frame_summary.csv",
    "detection_persistence.csv",
    "temporal_smoothed_windows.csv",
    "temporal_smoothed_trends.csv",
    "temporal_trend_summary.csv",
    "temporal_final_report.csv",
    "positive_video_evaluation.csv",
    "positive_video_evaluation_summary.csv",
    "positive_video_evaluation_report.txt",
]

missing = [name for name in required if not (OUT / name).exists()]
report = pd.read_csv(OUT / "temporal_final_report.csv")
trends = pd.read_csv(OUT / "temporal_smoothed_trends.csv")

check = report.merge(
    trends,
    on=["video_name", "class", "smoothed_trend"],
    how="left",
    indicator=True
)

trend_match = (check["_merge"] == "both").all()
zero_rows = report[report["detected_frames"] == 0]
zero_status_ok = zero_rows["final_temporal_status"].eq("No detections").all()

evaluation = pd.read_csv(OUT / "positive_video_evaluation.csv")
video_count = evaluation["video_name"].nunique()

passed = (
    not missing
    and trend_match
    and zero_status_ok
    and len(report) == 4
)

summary = f"""FireWatch AI - Notebook 05 Completion Check
Required outputs present: {len(required) - len(missing)}/{len(required)}
Trend consistency: {'PASS' if trend_match else 'FAIL'}
Zero-detection status: {'PASS' if zero_status_ok else 'FAIL'}
Detailed temporal report rows: {len(report)}
Positive-video evaluation videos saved: {video_count}
Overall validation: {'PASS' if passed else 'CHECK REQUIRED'}

Limitation: Temporal trends are preliminary visual estimates,
not proof of physical fire progression. Evaluation is limited
to the currently processed videos.
"""

(OUT / "notebook05_completion_check.txt").write_text(summary)

print(summary)

FireWatch AI - Notebook 05 Completion Check
Required outputs present: 10/10
Trend consistency: PASS
Zero-detection status: PASS
Detailed temporal report rows: 4
Positive-video evaluation videos saved: 13
Overall validation: PASS

Limitation: Temporal trends are preliminary visual estimates,
not proof of physical fire progression. Evaluation is limited
to the currently processed videos.

